# Marketplace Growth & Customer Experience Analysis

## C2. Data Cleaning and Standardisation

### Objective

This notebook prepares the raw Olist datasets for final validation and database loading.

The potential data-quality issues were identified in `C1_data_audit.ipynb`. Therefore, this notebook does not repeat the complete audit. It focuses on:

- Applying justified structural and formatting corrections.
- Preserving the original raw DataFrames.
- Retaining missing, unusual, or incomplete business records when no reliable correction is available.
- Measuring the impact of every applied transformation.
- Producing cleaned DataFrames for `C3_data_validation.ipynb`.

The cleaning approach is:

> **Use evidence from the audit, make the smallest justified change, and preserve business meaning.**

No statistical outlier, missing business value, or unusual record is removed solely because it appears abnormal.

## Cleaning Boundaries

This notebook may perform:

- Column-name correction
- Data-type standardisation
- Timestamp conversion
- ZIP-prefix formatting
- Text trimming
- Empty-string standardisation
- Removal of confirmed exact duplicate rows

This notebook will not perform:

- Missing-value imputation without a defensible source
- Statistical outlier removal
- Customer segmentation
- Revenue or delivery feature engineering
- Aggregation to analytical grains
- Correction of uncertain business timestamps
- Fabrication of missing category translations


In [1]:
import pandas as pd

from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

## 2. Load Raw Source Datasets

The raw CSV files are loaded without transformations.

This allows the notebook to run independently while preserving the original source representation.

In [2]:
DATA_PATH = Path("Data")

FILE_MAP = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

missing_files = [
    filename
    for filename in FILE_MAP.values()
    if not (DATA_PATH / filename).exists()
]

if missing_files:
    raise FileNotFoundError(
        "The following source files were not found:\n"
        + "\n".join(missing_files)
    )

datasets_raw = {
    table_name: pd.read_csv(
        DATA_PATH / filename,
        low_memory=False,
    )
    for table_name, filename in FILE_MAP.items()
}

print(f"Datasets loaded successfully: {len(datasets_raw)}")

Datasets loaded successfully: 9


In [3]:
source_inventory = pd.DataFrame(
    [
        {
            "table_name": table_name,
            "rows": len(dataframe),
            "columns": dataframe.shape[1],
        }
        for table_name, dataframe in datasets_raw.items()
    ]
)

display(source_inventory)

,table_name,rows,columns
0,customers,99441,5
1,orders,99441,8
2,order_items,112650,7
3,payments,103886,5
4,reviews,99224,7
5,products,32951,9
6,sellers,3095,4
7,geolocation,1000163,5
8,category_translation,71,2


## 3. Create Working Copies

All cleaning actions are applied to independent working copies.

The `datasets_raw` dictionary remains unchanged and represents the original source layer. The `datasets_clean` dictionary represents the cleaned layer.

In [4]:
datasets_clean = {
    table_name: dataframe.copy()
    for table_name, dataframe in datasets_raw.items()
}

baseline_rows = {
    table_name: len(dataframe)
    for table_name, dataframe in datasets_raw.items()
}

baseline_columns = {
    table_name: dataframe.shape[1]
    for table_name, dataframe in datasets_raw.items()
}

cleaning_log = []

print("Working copies created.")

Working copies created.


## 4. Structural Standardisation

Structural standardisation improves the usability and consistency of the cleaned datasets without changing their business meaning.

The following actions are performed:

1. Correct misspelled product-column names.
2. Convert timestamp fields to datetime.
3. Convert text fields from generic object types to pandas string types.
4. Represent ZIP-code prefixes as five-character identifiers.
5. Convert validated integer-valued product attributes to nullable integers.

### 4.1 Correct Product Column Names

#### Evidence

The products source file contains two misspelled column names:

- `product_name_lenght`
- `product_description_lenght`

#### Decision

Correct the spelling only in the cleaned layer:

- `product_name_length`
- `product_description_length`

The raw source columns remain unchanged.

In [5]:
product_column_rename_map = {
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length",
}

missing_source_columns = [
    column
    for column in product_column_rename_map
    if column not in datasets_clean["products"].columns
]

if missing_source_columns:
    raise KeyError(
        "Expected source columns were not found: "
        + ", ".join(missing_source_columns)
    )

datasets_clean["products"] = datasets_clean["products"].rename(
    columns=product_column_rename_map
)

cleaning_log.append(
    {
        "area": "Column naming",
        "table_name": "products",
        "columns": ", ".join(product_column_rename_map.keys()),
        "action": "Corrected misspelled column names",
        "impact": "2 columns renamed; no row values changed",
    }
)

display(
    pd.DataFrame(
        {
            "source_column": product_column_rename_map.keys(),
            "cleaned_column": product_column_rename_map.values(),
        }
    )
)

,source_column,cleaned_column
0,product_name_lenght,product_name_length
1,product_description_lenght,product_description_length


In [6]:
assert "product_name_lenght" not in datasets_clean["products"].columns
assert "product_description_lenght" not in datasets_clean["products"].columns

assert "product_name_length" in datasets_clean["products"].columns
assert "product_description_length" in datasets_clean["products"].columns

print("Product column-name standardisation completed.")

Product column-name standardisation completed.


### 4.2 Convert Timestamp Columns

#### Evidence

The audit found that the order, order-item, and review timestamps were loaded as text. Their non-null values were successfully parseable.

#### Decision

Convert the eight timestamp columns to pandas datetime types.

Missing timestamps remain missing. No date is estimated, replaced, or corrected.

In [7]:
datetime_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "order_items": [
        "shipping_limit_date",
    ],
    "reviews": [
        "review_creation_date",
        "review_answer_timestamp",
    ],
}

TIMESTAMP_FORMAT = "%Y-%m-%d %H:%M:%S"

datetime_missing_before = {
    (table_name, column): int(
        datasets_clean[table_name][column]
        .isna()
        .sum()
    )
    for table_name, columns in datetime_columns.items()
    for column in columns
}

for table_name, columns in datetime_columns.items():
    for column in columns:
        datasets_clean[table_name][column] = (
            pd.to_datetime(
                datasets_clean[table_name][column],
                format=TIMESTAMP_FORMAT,
                errors="raise",
            )
        )

cleaning_log.append(
    {
        "area": "Data types",
        "table_name": "orders, order_items, reviews",
        "columns": "8 timestamp columns",
        "action": (
            "Converted text timestamps to datetime "
            "using the validated source format"
        ),
        "impact": (
            "Missing values preserved; no timestamps "
            "imputed or corrected"
        ),
    }
)

In [8]:
datetime_verification = []

for table_name, columns in datetime_columns.items():
    for column in columns:
        missing_after = int(
            datasets_clean[table_name][column].isna().sum()
        )

        datetime_verification.append(
            {
                "table_name": table_name,
                "column_name": column,
                "cleaned_dtype": str(
                    datasets_clean[table_name][column].dtype
                ),
                "missing_before": datetime_missing_before[
                    (table_name, column)
                ],
                "missing_after": missing_after,
                "missing_count_preserved": (
                    datetime_missing_before[(table_name, column)]
                    == missing_after
                ),
            }
        )

datetime_verification = pd.DataFrame(datetime_verification)

display(datetime_verification)

,table_name,column_name,cleaned_dtype,missing_before,missing_after,missing_count_preserved
0,orders,order_purchase_timestamp,datetime64[ns],0,0,True
1,orders,order_approved_at,datetime64[ns],160,160,True
2,orders,order_delivered_carrier_date,datetime64[ns],1783,1783,True
3,orders,order_delivered_customer_date,datetime64[ns],2965,2965,True
4,orders,order_estimated_delivery_date,datetime64[ns],0,0,True
5,order_items,shipping_limit_date,datetime64[ns],0,0,True
6,reviews,review_creation_date,datetime64[ns],0,0,True
7,reviews,review_answer_timestamp,datetime64[ns],0,0,True


In [9]:
assert datetime_verification["missing_count_preserved"].all()

assert all(
    pd.api.types.is_datetime64_any_dtype(
        datasets_clean[table_name][column]
    )
    for table_name, columns in datetime_columns.items()
    for column in columns
)

print("Timestamp conversion completed without changing missingness.")

Timestamp conversion completed without changing missingness.


### 4.3 Standardise Text Data Types

#### Decision

After converting the timestamp fields, the remaining columns stored as generic pandas `object` types are textual fields.

These columns are converted to pandas' nullable `string` type.

This changes the representation, not the underlying text values.

In [10]:
string_conversion_summary = []

for table_name, dataframe in datasets_clean.items():
    object_columns = dataframe.select_dtypes(
        include="object"
    ).columns.tolist()

    for column in object_columns:
        datasets_clean[table_name][column] = (
            datasets_clean[table_name][column]
            .astype("string")
        )

    string_conversion_summary.append(
        {
            "table_name": table_name,
            "columns_converted": len(object_columns),
            "column_names": ", ".join(object_columns),
        }
    )

string_conversion_summary = pd.DataFrame(
    string_conversion_summary
)

cleaning_log.append(
    {
        "area": "Data types",
        "table_name": "All applicable tables",
        "columns": "Remaining object columns",
        "action": "Converted object columns to nullable string dtype",
        "impact": (
            f"{string_conversion_summary['columns_converted'].sum()} "
            "columns standardised"
        ),
    }
)

display(string_conversion_summary)

,table_name,columns_converted,column_names
0,customers,4,"customer_id, customer_unique_id, customer_city..."
1,orders,3,"order_id, customer_id, order_status"
2,order_items,3,"order_id, product_id, seller_id"
3,payments,2,"order_id, payment_type"
4,reviews,4,"review_id, order_id, review_comment_title, rev..."
5,products,2,"product_id, product_category_name"
6,sellers,3,"seller_id, seller_city, seller_state"
7,geolocation,2,"geolocation_city, geolocation_state"
8,category_translation,2,"product_category_name, product_category_name_e..."


### 4.4 Standardise ZIP-Code Prefixes

#### Evidence

Customer, seller, and geolocation ZIP-code prefixes are identifiers rather than quantities.

Numeric storage can omit leading zeros and creates an inappropriate mathematical representation.

#### Decision

Convert each ZIP-code prefix into a five-character string using leading-zero padding where required.

No location record is changed beyond its identifier representation.

In [11]:
zip_columns = {
    "customers": "customer_zip_code_prefix",
    "sellers": "seller_zip_code_prefix",
    "geolocation": "geolocation_zip_code_prefix",
}


def standardise_zip_prefix(series):
    """Convert a ZIP-prefix series to five-character strings."""

    numeric_values = pd.to_numeric(
        series,
        errors="raise",
    )

    non_null_values = numeric_values.dropna()

    if not (non_null_values % 1 == 0).all():
        raise ValueError(
            "ZIP-code prefixes contain non-integer values."
        )

    if not non_null_values.between(0, 99999).all():
        raise ValueError(
            "ZIP-code prefixes contain values outside "
            "the expected five-digit range."
        )

    return (
        numeric_values
        .astype("Int64")
        .astype("string")
        .str.zfill(5)
    )

In [12]:
zip_standardisation_summary = []

for table_name, column in zip_columns.items():
    before = datasets_clean[table_name][column].copy()

    datasets_clean[table_name][column] = standardise_zip_prefix(
        datasets_clean[table_name][column]
    )

    after = datasets_clean[table_name][column]

    changed_representation = int(
        (
            before.astype("string")
            != after
        )
        .fillna(False)
        .sum()
    )

    zip_standardisation_summary.append(
        {
            "table_name": table_name,
            "column_name": column,
            "rows": len(after),
            "representation_changed": changed_representation,
            "missing_values": int(after.isna().sum()),
        }
    )

zip_standardisation_summary = pd.DataFrame(
    zip_standardisation_summary
)

cleaning_log.append(
    {
        "area": "Identifier formatting",
        "table_name": "customers, sellers, geolocation",
        "columns": ", ".join(zip_columns.values()),
        "action": "Converted ZIP prefixes to five-character strings",
        "impact": (
            f"{zip_standardisation_summary['representation_changed'].sum():,} "
            "values received a standardised representation"
        ),
    }
)

display(zip_standardisation_summary)

,table_name,column_name,rows,representation_changed,missing_values
0,customers,customer_zip_code_prefix,99441,23995,0
1,sellers,seller_zip_code_prefix,3095,1027,0
2,geolocation,geolocation_zip_code_prefix,1000163,245733,0


In [13]:
for table_name, column in zip_columns.items():
    zip_values = (
        datasets_clean[table_name][column]
        .dropna()
    )

    assert zip_values.str.fullmatch(r"\d{5}").all()

print("All non-null ZIP-code prefixes use five-digit string format.")

All non-null ZIP-code prefixes use five-digit string format.


### 4.5 Convert Integer-Valued Product Attributes

#### Evidence

Several product attributes are loaded as floating-point values because their columns contain missing records.

The project database defines the following fields as integer-valued attributes:

- Product-name length
- Product-description length
- Product-photo quantity
- Product weight in grams
- Product length in centimetres
- Product height in centimetres
- Product width in centimetres

#### Decision

Verify that every non-null value is a whole number and convert the columns to pandas nullable integer type (`Int64`).

Existing missing values remain missing. The notebook will stop if a non-integer measurement is found rather than rounding it.

In [14]:
product_integer_columns = [
    "product_name_length",
    "product_description_length",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]

product_missing_before = {
    column: int(
        datasets_clean["products"][column]
        .isna()
        .sum()
    )
    for column in product_integer_columns
}

for column in product_integer_columns:
    numeric_values = pd.to_numeric(
        datasets_clean["products"][column],
        errors="raise",
    )

    non_null_values = numeric_values.dropna()

    if not non_null_values.mod(1).eq(0).all():
        raise ValueError(
            f"{column} contains non-integer values and "
            "cannot be converted to Int64 without rounding."
        )

    datasets_clean["products"][column] = (
        numeric_values.astype("Int64")
    )

product_integer_verification = pd.DataFrame(
    [
        {
            "column_name": column,
            "cleaned_dtype": str(
                datasets_clean["products"][column].dtype
            ),
            "missing_before": product_missing_before[
                column
            ],
            "missing_after": int(
                datasets_clean["products"][column]
                .isna()
                .sum()
            ),
        }
        for column in product_integer_columns
    ]
)

cleaning_log.append(
    {
        "area": "Data types",
        "table_name": "products",
        "columns": ", ".join(
            product_integer_columns
        ),
        "action": (
            "Validated whole-number values and converted "
            "integer-valued product attributes to Int64"
        ),
        "impact": (
            f"{len(product_integer_columns)} columns "
            "converted; missing values preserved"
        ),
    }
)

display(product_integer_verification)

,column_name,cleaned_dtype,missing_before,missing_after
0,product_name_length,Int64,610,610
1,product_description_length,Int64,610,610
2,product_photos_qty,Int64,610,610
3,product_weight_g,Int64,2,2
4,product_length_cm,Int64,2,2
5,product_height_cm,Int64,2,2
6,product_width_cm,Int64,2,2


In [15]:
assert (
    product_integer_verification["missing_before"]
    == product_integer_verification["missing_after"]
).all()

assert all(
    str(
        datasets_clean["products"][column].dtype
    )
    == "Int64"
    for column in product_integer_columns
)

print(
    "Integer-valued product attributes were converted "
    "without changing missingness."
)

Integer-valued product attributes were converted without changing missingness.


## 5. Text Standardisation

### Evidence

The audit identified a small number of values with leading or trailing whitespace, particularly in review text and geographic fields.

Whitespace-only review values do not contain meaningful text.

### Decision

For selected descriptive and categorical fields:

- Remove leading and trailing whitespace.
- Convert resulting empty strings to missing values.
- Preserve internal spacing, accents, punctuation, capitalisation, and Portuguese text.
- Do not alter numeric-looking city values because their correct replacement cannot be inferred.

In [16]:
text_columns = {
    "customers": [
        "customer_city",
        "customer_state",
    ],
    "sellers": [
        "seller_city",
        "seller_state",
    ],
    "geolocation": [
        "geolocation_city",
        "geolocation_state",
    ],
    "orders": [
        "order_status",
    ],
    "payments": [
        "payment_type",
    ],
    "products": [
        "product_category_name",
    ],
    "reviews": [
        "review_comment_title",
        "review_comment_message",
    ],
    "category_translation": [
        "product_category_name",
        "product_category_name_english",
    ],
}

text_cleaning_summary = []

for table_name, columns in text_columns.items():
    for column in columns:
        before = datasets_clean[table_name][column].copy()

        stripped = before.str.strip()

        whitespace_changed = int(
            (
                before.notna()
                & stripped.notna()
                & before.ne(stripped)
            ).sum()
        )

        empty_to_missing = int(
            stripped.eq("")
            .fillna(False)
            .sum()
        )

        datasets_clean[table_name][column] = (
            stripped.replace("", pd.NA)
        )

        text_cleaning_summary.append(
            {
                "table_name": table_name,
                "column_name": column,
                "whitespace_values_changed": whitespace_changed,
                "empty_values_converted_to_missing": empty_to_missing,
            }
        )

text_cleaning_summary = pd.DataFrame(
    text_cleaning_summary
)

cleaning_log.append(
    {
        "area": "Text formatting",
        "table_name": "Selected text fields",
        "columns": "Cities, states, categories, statuses and review text",
        "action": (
            "Trimmed surrounding whitespace and converted "
            "empty strings to missing values"
        ),
        "impact": (
            f"{text_cleaning_summary['whitespace_values_changed'].sum():,} "
            "values trimmed; "
            f"{text_cleaning_summary['empty_values_converted_to_missing'].sum():,} "
            "empty values converted to missing"
        ),
    }
)

display(
    text_cleaning_summary.query(
        "whitespace_values_changed > 0 "
        "or empty_values_converted_to_missing > 0"
    )
)

,table_name,column_name,whitespace_values_changed,empty_values_converted_to_missing
4,geolocation,geolocation_city,1,0
9,reviews,review_comment_title,1998,2
10,reviews,review_comment_message,9451,27


In [17]:
for table_name, columns in text_columns.items():
    for column in columns:
        non_null_values = (
            datasets_clean[table_name][column]
            .dropna()
        )

        assert non_null_values.eq(
            non_null_values.str.strip()
        ).all()

        assert not non_null_values.eq("").any()

print("Text standardisation completed.")

Text standardisation completed.


## 6. Exact Duplicate Handling

### 6.1 Geolocation Exact Duplicates

#### Evidence

The geolocation dataset contains rows in which all five source fields are identical.

These exact duplicates provide no additional location information and can overweight identical observations during later geographic summarisation.

#### Decision

Remove only fully identical geolocation rows.

Records sharing a ZIP-code prefix but having different coordinates, cities, or states are retained because they are not exact duplicates.

This operation does not create one row per ZIP code. Geographic aggregation belongs to the later database or analytical modelling stage.

In [18]:
geolocation_rows_before = len(
    datasets_clean["geolocation"]
)

geolocation_duplicates_before = int(
    datasets_clean["geolocation"]
    .duplicated()
    .sum()
)

datasets_clean["geolocation"] = (
    datasets_clean["geolocation"]
    .drop_duplicates()
    .reset_index(drop=True)
)

geolocation_rows_after = len(
    datasets_clean["geolocation"]
)

geolocation_rows_removed = (
    geolocation_rows_before
    - geolocation_rows_after
)

cleaning_log.append(
    {
        "area": "Duplicate records",
        "table_name": "geolocation",
        "columns": "All columns",
        "action": "Removed fully identical rows",
        "impact": (
            f"{geolocation_rows_removed:,} exact duplicate "
            "rows removed"
        ),
    }
)

geolocation_duplicate_summary = pd.DataFrame(
    [
        {
            "rows_before": geolocation_rows_before,
            "exact_duplicates_detected": (
                geolocation_duplicates_before
            ),
            "rows_removed": geolocation_rows_removed,
            "rows_after": geolocation_rows_after,
            "exact_duplicates_remaining": int(
                datasets_clean["geolocation"]
                .duplicated()
                .sum()
            ),
        }
    ]
)

display(geolocation_duplicate_summary)

,rows_before,exact_duplicates_detected,rows_removed,rows_after,exact_duplicates_remaining
0,1000163,261831,261831,738332,0


In [19]:
assert (
    geolocation_rows_removed
    == geolocation_duplicates_before
)

assert not datasets_clean["geolocation"].duplicated().any()

print("Exact geolocation duplicates removed successfully.")

Exact geolocation duplicates removed successfully.


### 6.2 Repeated Review Identifiers

The audit found repeated `review_id` values. However, the combination of `review_id` and `order_id` remains unique.

#### Decision

Retain all review records.

A repeated `review_id` is not treated as an exact duplicate when it is connected to a different order. Removing such records would change the observed review grain without sufficient evidence.

In [20]:
reviews_clean = datasets_clean["reviews"]

review_key_summary = pd.DataFrame(
    [
        {
            "check": "Rows with repeated review_id",
            "affected_rows": int(
                reviews_clean["review_id"]
                .duplicated(keep=False)
                .sum()
            ),
        },
        {
            "check": "Duplicate review_id + order_id combinations",
            "affected_rows": int(
                reviews_clean
                .duplicated(
                    subset=["review_id", "order_id"]
                )
                .sum()
            ),
        },
        {
            "check": "Exact duplicate review rows",
            "affected_rows": int(
                reviews_clean.duplicated().sum()
            ),
        },
    ]
)

display(review_key_summary)

,check,affected_rows
0,Rows with repeated review_id,1603
1,Duplicate review_id + order_id combinations,0
2,Exact duplicate review rows,0


In [21]:
assert not reviews_clean.duplicated(
    subset=["review_id", "order_id"]
).any()

assert not reviews_clean.duplicated().any()

print(
    "Review records retained because the composite key "
    "remains unique."
)

Review records retained because the composite key remains unique.


## 7. Missing, Unusual and Unresolved Values

Not every audit finding requires a transformation.

The following records are retained:

- Missing order lifecycle timestamps
- Missing review titles and messages
- Missing product categories and metadata
- Missing product dimensions
- `not_defined` payment types
- Zero payment values or instalment counts
- Zero product weights
- Repeated review identifiers with unique composite keys
- Product categories without English translations
- Chronological anomalies without a verifiable replacement
- Statistically extreme prices, freight charges, payment values, weights, and dimensions

These values may require filters or interpretation during analysis, but altering them here would require unsupported assumptions.

### Decision on Missing Values

No missing business value is imputed in this notebook.

Examples:

- A missing delivery date may correspond to an incomplete or cancelled order.
- A missing review comment may mean the customer submitted only a score.
- Missing product metadata cannot be reconstructed reliably.
- A missing English category translation does not justify inventing a translation.

Missing values are therefore preserved unless a formatting operation converts a confirmed blank string into a proper missing value.

In [22]:
customers_clean = datasets_clean["customers"]
orders_clean = datasets_clean["orders"]
order_items_clean = datasets_clean["order_items"]
payments_clean = datasets_clean["payments"]
reviews_clean = datasets_clean["reviews"]
products_clean = datasets_clean["products"]
sellers_clean = datasets_clean["sellers"]
geolocation_clean = datasets_clean["geolocation"]
category_translation_clean = datasets_clean[
    "category_translation"
]

product_dimension_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]

translated_categories = set(
    category_translation_clean[
        "product_category_name"
    ].dropna()
)

product_categories = set(
    products_clean[
        "product_category_name"
    ].dropna()
)

untranslated_categories = (
    product_categories
    - translated_categories
)

review_order_timeline = (
    reviews_clean[
        [
            "review_id",
            "order_id",
            "review_creation_date",
        ]
    ]
    .merge(
        orders_clean[
            [
                "order_id",
                "order_purchase_timestamp",
            ]
        ],
        on="order_id",
        how="left",
        validate="many_to_one",
    )
)

review_order_timeline["survey_sent_date"] = (
    review_order_timeline[
        "review_creation_date"
    ].dt.normalize()
)

review_order_timeline["purchase_date"] = (
    review_order_timeline[
        "order_purchase_timestamp"
    ].dt.normalize()
)

retained_value_summary = pd.DataFrame(
    [
        {
            "table_name": "orders",
            "condition": "Missing order_approved_at",
            "affected_rows": int(
                orders_clean["order_approved_at"]
                .isna()
                .sum()
            ),
            "decision": "Retain",
        },
        {
            "table_name": "orders",
            "condition": (
                "Missing order_delivered_carrier_date"
            ),
            "affected_rows": int(
                orders_clean[
                    "order_delivered_carrier_date"
                ]
                .isna()
                .sum()
            ),
            "decision": "Retain",
        },
        {
            "table_name": "orders",
            "condition": (
                "Missing order_delivered_customer_date"
            ),
            "affected_rows": int(
                orders_clean[
                    "order_delivered_customer_date"
                ]
                .isna()
                .sum()
            ),
            "decision": "Retain",
        },
        {
            "table_name": "reviews",
            "condition": "Missing review title",
            "affected_rows": int(
                reviews_clean[
                    "review_comment_title"
                ]
                .isna()
                .sum()
            ),
            "decision": "Retain",
        },
        {
            "table_name": "reviews",
            "condition": "Missing review message",
            "affected_rows": int(
                reviews_clean[
                    "review_comment_message"
                ]
                .isna()
                .sum()
            ),
            "decision": "Retain",
        },
        {
            "table_name": "products",
            "condition": "Missing product category",
            "affected_rows": int(
                products_clean[
                    "product_category_name"
                ]
                .isna()
                .sum()
            ),
            "decision": "Retain",
        },
        {
            "table_name": "products",
            "condition": (
                "At least one missing physical dimension"
            ),
            "affected_rows": int(
                products_clean[
                    product_dimension_columns
                ]
                .isna()
                .any(axis=1)
                .sum()
            ),
            "decision": "Retain",
        },
        {
            "table_name": "payments",
            "condition": "payment_type = not_defined",
            "affected_rows": int(
                payments_clean["payment_type"]
                .eq("not_defined")
                .sum()
            ),
            "decision": "Retain",
        },
        {
            "table_name": "payments",
            "condition": "payment_value <= 0",
            "affected_rows": int(
                payments_clean["payment_value"]
                .le(0)
                .sum()
            ),
            "decision": "Retain",
        },
        {
            "table_name": "payments",
            "condition": "payment_installments <= 0",
            "affected_rows": int(
                payments_clean[
                    "payment_installments"
                ]
                .le(0)
                .sum()
            ),
            "decision": "Retain",
        },
        {
            "table_name": "products",
            "condition": "product_weight_g <= 0",
            "affected_rows": int(
                products_clean["product_weight_g"]
                .le(0)
                .sum()
            ),
            "decision": "Retain",
        },
        {
            "table_name": "reviews",
            "condition": "Rows with repeated review_id",
            "affected_rows": int(
                reviews_clean["review_id"]
                .duplicated(keep=False)
                .sum()
            ),
            "decision": "Retain",
        },
        {
            "table_name": "products",
            "condition": (
                "Products in untranslated categories"
            ),
            "affected_rows": int(
                products_clean[
                    "product_category_name"
                ]
                .isin(untranslated_categories)
                .sum()
            ),
            "decision": "Retain Portuguese category",
        },
        {
            "table_name": "orders",
            "condition": (
                "Carrier timestamp before purchase"
            ),
            "affected_rows": int(
                (
                    orders_clean[
                        "order_delivered_carrier_date"
                    ].notna()
                    & (
                        orders_clean[
                            "order_delivered_carrier_date"
                        ]
                        < orders_clean[
                            "order_purchase_timestamp"
                        ]
                    )
                ).sum()
            ),
            "decision": (
                "Retain and flag during analysis"
            ),
        },
        {
            "table_name": "orders",
            "condition": (
                "Customer delivery before carrier handoff"
            ),
            "affected_rows": int(
                (
                    orders_clean[
                        "order_delivered_customer_date"
                    ].notna()
                    & orders_clean[
                        "order_delivered_carrier_date"
                    ].notna()
                    & (
                        orders_clean[
                            "order_delivered_customer_date"
                        ]
                        < orders_clean[
                            "order_delivered_carrier_date"
                        ]
                    )
                ).sum()
            ),
            "decision": (
                "Retain and flag during analysis"
            ),
        },
        {
            "table_name": "order_items",
            "condition": (
                "Shipping-limit date in 2019 or later"
            ),
            "affected_rows": int(
                (
                    order_items_clean[
                        "shipping_limit_date"
                    ].notna()
                    & (
                        order_items_clean[
                            "shipping_limit_date"
                        ]
                        >= pd.Timestamp("2019-01-01")
                    )
                ).sum()
            ),
            "decision": (
                "Retain and flag during analysis"
            ),
        },
        {
            "table_name": "reviews",
            "condition": (
                "Satisfaction survey sent before "
                "purchase date"
            ),
            "affected_rows": int(
                (
                    review_order_timeline[
                        "survey_sent_date"
                    ].notna()
                    & review_order_timeline[
                        "purchase_date"
                    ].notna()
                    & (
                        review_order_timeline[
                            "survey_sent_date"
                        ]
                        < review_order_timeline[
                            "purchase_date"
                        ]
                    )
                ).sum()
            ),
            "decision": (
                "Retain and flag during analysis"
            ),
        },
    ]
)

display(retained_value_summary)

,table_name,condition,affected_rows,decision
0,orders,Missing order_approved_at,160,Retain
1,orders,Missing order_delivered_carrier_date,1783,Retain
2,orders,Missing order_delivered_customer_date,2965,Retain
3,reviews,Missing review title,87658,Retain
4,reviews,Missing review message,58274,Retain
5,products,Missing product category,610,Retain
6,products,At least one missing physical dimension,2,Retain
7,payments,payment_type = not_defined,3,Retain
8,payments,payment_value <= 0,9,Retain
9,payments,payment_installments <= 0,2,Retain


### Interpretation

The table above documents deliberate non-actions.

Retaining these records is not an oversight. It reflects the absence of reliable replacement values and avoids changing valid or potentially valid business events.

Later analytical queries must apply metric-specific eligibility rules. For example, delivery-duration calculations should require valid purchase and delivery timestamps rather than deleting incomplete orders from the cleaned dataset.

## 8. Cleaned DataFrames

The following DataFrames represent the cleaned versions of the nine source tables:

- `customers_clean`
- `orders_clean`
- `order_items_clean`
- `payments_clean`
- `reviews_clean`
- `products_clean`
- `sellers_clean`
- `geolocation_clean`
- `category_translation_clean`

No intermediate cleaned CSV files are created.

In [23]:
cleaned_datasets = {
    "customers": customers_clean,
    "orders": orders_clean,
    "order_items": order_items_clean,
    "payments": payments_clean,
    "reviews": reviews_clean,
    "products": products_clean,
    "sellers": sellers_clean,
    "geolocation": geolocation_clean,
    "category_translation": category_translation_clean,
}

cleaned_inventory = pd.DataFrame(
    [
        {
            "table_name": table_name,
            "rows": len(dataframe),
            "columns": dataframe.shape[1],
            "missing_cells": int(
                dataframe.isna().sum().sum()
            ),
        }
        for table_name, dataframe
        in cleaned_datasets.items()
    ]
)

display(cleaned_inventory)

,table_name,rows,columns,missing_cells
0,customers,99441,5,0
1,orders,99441,8,4908
2,order_items,112650,7,0
3,payments,103886,5,0
4,reviews,99224,7,145932
5,products,32951,9,2448
6,sellers,3095,4,0
7,geolocation,738332,5,0
8,category_translation,71,2,0


## 9. Cleaning Log

The cleaning log records transformations that were actually applied.

Values intentionally retained are documented separately in the retained-value summary.

In [24]:
cleaning_log_df = pd.DataFrame(cleaning_log)

display(cleaning_log_df)

,area,table_name,columns,action,impact
0,Column naming,products,"product_name_lenght, product_description_lenght",Corrected misspelled column names,2 columns renamed; no row values changed
1,Data types,"orders, order_items, reviews",8 timestamp columns,Converted text timestamps to datetime using th...,Missing values preserved; no timestamps impute...
2,Data types,All applicable tables,Remaining object columns,Converted object columns to nullable string dtype,25 columns standardised
3,Identifier formatting,"customers, sellers, geolocation","customer_zip_code_prefix, seller_zip_code_pref...",Converted ZIP prefixes to five-character strings,"270,755 values received a standardised represe..."
4,Data types,products,"product_name_length, product_description_lengt...",Validated whole-number values and converted in...,7 columns converted; missing values preserved
5,Text formatting,Selected text fields,"Cities, states, categories, statuses and revie...",Trimmed surrounding whitespace and converted e...,"11,450 values trimmed; 29 empty values convert..."
6,Duplicate records,geolocation,All columns,Removed fully identical rows,"261,831 exact duplicate rows removed"


# 10. Post-Cleaning Verification

This section verifies the direct effects of the cleaning transformations.

The comprehensive database-readiness assessment—including full relationship, grain, and business-rule validation—will be performed in `C3_data_validation.ipynb`.

In [25]:
row_reconciliation = pd.DataFrame(
    [
        {
            "table_name": table_name,
            "raw_rows": baseline_rows[table_name],
            "cleaned_rows": len(
                cleaned_datasets[table_name]
            ),
            "row_difference": (
                len(cleaned_datasets[table_name])
                - baseline_rows[table_name]
            ),
            "raw_columns": baseline_columns[table_name],
            "cleaned_columns": (
                cleaned_datasets[table_name].shape[1]
            ),
            "column_difference": (
                cleaned_datasets[table_name].shape[1]
                - baseline_columns[table_name]
            ),
        }
        for table_name in cleaned_datasets
    ]
)

display(row_reconciliation)

,table_name,raw_rows,cleaned_rows,row_difference,raw_columns,cleaned_columns,column_difference
0,customers,99441,99441,0,5,5,0
1,orders,99441,99441,0,8,8,0
2,order_items,112650,112650,0,7,7,0
3,payments,103886,103886,0,5,5,0
4,reviews,99224,99224,0,7,7,0
5,products,32951,32951,0,9,9,0
6,sellers,3095,3095,0,4,4,0
7,geolocation,1000163,738332,-261831,5,5,0
8,category_translation,71,71,0,2,2,0


In [26]:
non_geolocation_changes = row_reconciliation.loc[
    (
        row_reconciliation["table_name"]
        != "geolocation"
    )
    & (
        row_reconciliation["row_difference"]
        != 0
    )
]

assert non_geolocation_changes.empty

geolocation_difference = int(
    row_reconciliation.loc[
        row_reconciliation["table_name"]
        == "geolocation",
        "row_difference",
    ].iloc[0]
)

assert geolocation_difference == -geolocation_rows_removed

assert row_reconciliation[
    "column_difference"
].eq(0).all()

print("Row and column reconciliation passed.")

Row and column reconciliation passed.


## Key Preservation Check

Cleaning must not create missing or duplicate values in the proposed table keys.

In [27]:
key_definitions = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "order_items": [
        "order_id",
        "order_item_id",
    ],
    "payments": [
        "order_id",
        "payment_sequential",
    ],
    "reviews": [
        "review_id",
        "order_id",
    ],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "category_translation": [
        "product_category_name"
    ],
}

key_verification = []

for table_name, key_columns in key_definitions.items():
    dataframe = cleaned_datasets[table_name]

    key_verification.append(
        {
            "table_name": table_name,
            "key_columns": " + ".join(key_columns),
            "missing_key_rows": int(
                dataframe[key_columns]
                .isna()
                .any(axis=1)
                .sum()
            ),
            "duplicate_key_rows": int(
                dataframe
                .duplicated(
                    subset=key_columns
                )
                .sum()
            ),
        }
    )

key_verification = pd.DataFrame(
    key_verification
)

display(key_verification)

,table_name,key_columns,missing_key_rows,duplicate_key_rows
0,customers,customer_id,0,0
1,orders,order_id,0,0
2,order_items,order_id + order_item_id,0,0
3,payments,order_id + payment_sequential,0,0
4,reviews,review_id + order_id,0,0
5,products,product_id,0,0
6,sellers,seller_id,0,0
7,category_translation,product_category_name,0,0


In [28]:
assert key_verification["missing_key_rows"].eq(0).all()
assert key_verification["duplicate_key_rows"].eq(0).all()

print("Proposed keys remain complete and unique.")

Proposed keys remain complete and unique.


In [29]:
# Raw source names remain unchanged.

assert "product_name_lenght" in (
    datasets_raw["products"].columns
)

assert "product_description_lenght" in (
    datasets_raw["products"].columns
)


# Cleaned names use corrected spelling.

assert "product_name_length" in (
    products_clean.columns
)

assert "product_description_length" in (
    products_clean.columns
)


# All intended timestamp columns use datetime types.

for table_name, columns in datetime_columns.items():
    for column in columns:
        assert pd.api.types.is_datetime64_any_dtype(
            cleaned_datasets[table_name][column]
        )


# Integer-valued product attributes use nullable integers.

for column in product_integer_columns:
    assert (
        str(products_clean[column].dtype)
        == "Int64"
    )


# All ZIP prefixes contain five digits.

for table_name, column in zip_columns.items():
    assert (
        cleaned_datasets[table_name][column]
        .dropna()
        .str.fullmatch(r"\d{5}")
        .all()
    )


# Exact geolocation duplicates have been removed.

assert not geolocation_clean.duplicated().any()


# Target text fields contain no surrounding whitespace
# or empty strings.

for table_name, columns in text_columns.items():
    for column in columns:
        values = (
            cleaned_datasets[table_name][column]
            .dropna()
        )

        assert values.eq(
            values.str.strip()
        ).all()

        assert not values.eq("").any()


print("All transformation checks passed.")

All transformation checks passed.


# Final Cleaning Summary

The cleaning process applied a limited set of evidence-supported transformations:

1. Corrected two misspelled product column names in the cleaned layer.
2. Converted eight timestamp columns to datetime using the validated source format.
3. Converted textual object columns to pandas nullable string type.
4. Standardised customer, seller, and geolocation ZIP-code prefixes as five-character identifiers.
5. Converted seven validated integer-valued product attributes to nullable integer types.
6. Removed leading and trailing whitespace from selected text fields.
7. Standardised confirmed empty strings as missing values.
8. Removed fully identical duplicate rows from the geolocation table.

The following were intentionally retained:

- Missing lifecycle timestamps
- Missing review text
- Missing product metadata
- Unusual payment records
- Zero or extreme numerical values
- Repeated review identifiers with valid composite keys
- Untranslated Portuguese product categories
- Uncertain temporal anomalies
- Statistical outliers

No business value was imputed, fabricated, rounded, or corrected without sufficient supporting evidence.

The cleaned DataFrames are now ready for comprehensive validation in `C3_data_validation.ipynb` and subsequent database loading.